In [ ]:

# ─── Importações ─────────────────────────────────────────────────────────────

import pandas as pd, spacy

# Stop words em português fornecidas pelo spaCy
from spacy.lang.pt.stop_words import STOP_WORDS

# Classificadores supervisionados do scikit-learn
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# Vetorizadores de texto: contagem simples (BoW) e TF-IDF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# Seleção dos K melhores atributos usando testes estatísticos (qui-quadrado e F)
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
# Pipeline encadeia etapas de pré-processamento + modelo em um único objeto
from sklearn.pipeline import make_pipeline
# StandardScaler normaliza os atributos (com_mean=False evita erros em matrizes esparsas)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier


# ─── Carregamento do modelo de língua portuguesa ─────────────────────────────
# Desativa componentes desnecessários para acelerar o processamento:
# morfologia, sentenciador, parser sintático, regras de atributos e NER
pln = spacy.load('pt_core_news_sm', disable=['morphologizer', 'senter', 'parser', 'attribute_ruler', 'ner'])


# ─── Pré-processamento do corpus ─────────────────────────────────────────────
def preprocessing(corpus):
    """
    Recebe uma lista de textos e retorna a versão lematizada e sem stop words.
    Lematização: reduz palavras à sua forma base (ex.: 'correndo' → 'correr').
    """
    corpus_with_lemmas = []
    for sentence in corpus:
        # Converte para minúsculas antes de processar
        sentence = sentence.lower()
        # Processa o texto com o pipeline do spaCy (tokenização + lematização)
        doc = pln(sentence)
        # Filtra stop words e junta os lemas de volta em uma string
        sentence_with_lemmas = ' '.join([word.lemma_ for word in doc if not word.is_stop])
        corpus_with_lemmas.append(sentence_with_lemmas)

    return corpus_with_lemmas


# ─── Carga e preparação dos dados ────────────────────────────────────────────
# O dataset Buscapé contém avaliações de produtos e rótulos de polaridade (positivo/negativo)
df = pd.read_csv('buscape.csv').dropna()  # Remove linhas com valores nulos
corpus = df['review_text'].tolist()       # Textos das avaliações (entrada X)
y = df['polarity'].tolist()               # Rótulos de polaridade (saída y)

# Aplica lematização e remoção de stop words a todo o corpus
corpus = preprocessing(corpus)

# Divide em conjuntos de treino (75%) e teste (25%) com estratificação aleatória padrão
corpus_train, corpus_test, y_train, y_test = train_test_split(corpus, y)


# ─── Definição dos classificadores e seus grids de hiperparâmetros ───────────
# Cada tupla contém: (nome do classificador, instância do modelo, dicionário de params)
# O prefixo nos parâmetros (ex.: 'logisticregression__C') é exigido pelo Pipeline
# para identificar a qual etapa pertence cada hiperparâmetro.
classifiers = [
    ('Logistic Regression', LogisticRegression(random_state=42),
        {
            'logisticregression__C': [0.1, 1, 10],               # Força de regularização (menor = mais regularização)
            'logisticregression__tol': [1e-3, 1e-4, 1e-5],       # Tolerância de convergência
            'logisticregression__class_weight': [None, 'balanced'],  # 'balanced' compensa classes desiguais
        }),
    ('Linear SVM', LinearSVC(random_state=42),
        {
            'linearsvc__C': [0.1, 1, 10],
            'linearsvc__tol': [1e-3, 1e-4, 1e-5],
            'linearsvc__class_weight': [None, 'balanced'],
            'linearsvc__dual': ['auto']   # 'auto' escolhe formulação primal/dual automaticamente
        }),
    ('Random Forest', RandomForestClassifier(random_state=42),
        {
            'randomforestclassifier__n_estimators': [50, 100, 200],           # Número de árvores no ensemble
            'randomforestclassifier__max_depth': [10, 20, 30],                # Profundidade máxima de cada árvore
            'randomforestclassifier__criterion': ['gini', 'entropy'],         # Critério de divisão dos nós
            'randomforestclassifier__class_weight': [None, 'balanced']
        }),
    ('Gradient Boosting', GradientBoostingClassifier(random_state=42),
        {
            'gradientboostingclassifier__n_estimators': [50, 100, 200],       # Número de boosting rounds
            'gradientboostingclassifier__learning_rate': [0.01, 0.1],         # Taxa de aprendizado (shrinkage)
            'gradientboostingclassifier__max_depth': [3, 5, 7],               # Profundidade das árvores base
        }),
    ('Decision Tree', DecisionTreeClassifier(random_state=42),
        {
            'decisiontreeclassifier__max_depth': [10, 20, 30],
            'decisiontreeclassifier__criterion': ['gini', 'entropy'],
            'decisiontreeclassifier__class_weight': [None, 'balanced']
        })
]


# ─── Treinamento com Pipeline + GridSearchCV ──────────────────────────────────
for (classifier, model, parameters) in classifiers:
    print(f"- {classifier}")

    # Monta o pipeline com 4 etapas encadeadas:
    #   1. TfidfVectorizer  → converte texto em matriz TF-IDF com n-gramas de 1 a 3 palavras
    #   2. StandardScaler   → escala os atributos (with_mean=False obrigatório para matriz esparsa)
    #   3. SelectKBest      → retém apenas os K atributos mais informativos (estatística a definir)
    #   4. model            → classificador escolhido na iteração atual
    pipeline = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 3)),
        StandardScaler(with_mean=False, with_std=False),
        SelectKBest(), model)

    # Adiciona ao grid: qual função de pontuação usar na seleção de atributos
    # chi2 (qui-quadrado) funciona com valores não-negativos; f_classif usa ANOVA F-score
    parameters['selectkbest__score_func'] = [chi2, f_classif]
    # Número de atributos a manter: 100, 1000 ou 10000
    parameters['selectkbest__k'] = [100, 1000, 10000]

    # GridSearchCV testa todas as combinações de hiperparâmetros com validação cruzada de 3 folds
    # verbose=1 mostra o progresso no console
    grid_search = GridSearchCV(pipeline, parameters, cv=3, verbose=1)
    grid_search.fit(corpus_train, y_train)   # Treina com os melhores parâmetros encontrados

    # Avalia no conjunto de teste (dados nunca vistos durante o GridSearch)
    y_pred = grid_search.predict(corpus_test)

    # Exibe métricas por classe: precision, recall, F1-score e support
    print(classification_report(y_test, y_pred))
